# G0 · Ejecución cubo real

**Spec:** [`docs/spec_G0_codex_real_cube_execution.md`](../docs/spec_G0_codex_real_cube_execution.md)  |  **Bloque:** G · Caracterización  |  **Run de este set:** `ROXs12b_realigned`

Verifica la cadena de hash y compara con el legacy sobre el cubo real.

| | |
|---|---|
| **Entrada** | Productos A→F |
| **Salida (QC/productos)** | `stages/stage_g0_qc.json`, `tables/g0_legacy_comparison.csv` |
| **Consume aguas abajo** | G1–G5 |


## Qué hace G0 y por qué

G0 es la **entrada del bloque G (caracterización)**. Ejecuta la cadena multi-método **de extremo a extremo sobre el cubo real** por primera vez y deja constancia de cualquier fallo de infraestructura **antes** de tocar nada. Es *ejecutar, inspeccionar y corregir lo mínimo* — no reescribir.

**Verifica:**
- **`hash_chain` = pass**: la proveniencia de los productos es trazable (el sha del cubo de entrada casa a lo largo de la cadena).
- **`stat_verdict` = red**: STAT usable = False (factor 4.26, de A4/M5) — marcado, se usa ruido empírico aguas abajo.
- **`legacy_comparison`**: el run realineado vs el legacy ADP (flujo integrado por banda), **3 de 9 bandas flagged** (legacy: `ROXs12b_B_adp`; `n/d` = no hay ADP de archivo para este objeto y la comparación no aplica). Es **documentario** (las razones no son fiables donde el continuo es negativo, p.ej. post-Hα).
- **`frozen_criteria_untouched` = True**: no se tocaron los criterios congelados del gate para pasar (sin trampas).

**Deviación honesta** (open_issue): se reprodujo **retroactivamente** vía `scripts/run_g0.py` sobre un run `stage-*` existente, no en una rama `phase-g0` fresca — 'G0 cumplido en sustancia'. Con `hash_chain_ok=True`, el paquete es consistente en proveniencia → entrada a G1–G5.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python scripts/run_g0.py --run-id $RUN
```

Ligero.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_g0_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python scripts/run_g0.py --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_g0_qc.json', RUN_ID)
nb.show(qc, keys=['hash_chain_ok', 'stat_verdict', 'n_flagged', 'frozen_criteria_untouched'], title='G0')


## Los términos de este QC, en físico

| Término | Qué es | Por qué importa |
|---|---|---|
| `stat_verdict` | Si la extensión `STAT` del cubo real sirve como σ. | Es la pregunta de M5 hecha sobre el cubo definitivo: si el STAT subestima el ruido, **σ se mide siempre en controles** ([`docs/noise_model.md`](../../docs/noise_model.md)). |
| `hash_chain_ok` | Que el encadenado de sha256 entre etapas cuadre. | Garantiza que estos resultados salen de estos datos, sin productos mezclados de otra ejecución. |
| `frozen_criteria_untouched` | Que los umbrales y decisiones congelados sigan siendo los mismos. | Correr sobre datos reales **no** puede ir acompañado de aflojar un umbral: eso convertiría el criterio en una consecuencia del resultado. |
| `pytest_before` / `pytest_after` | La suite antes y después de la ejecución. | Deja constancia de que el código no cambió a mitad del proceso. |
| `legacy_comparison` | Contraste con la reducción histórica (ADP). | Sitúa la re-reducción propia frente a la del archivo. |


## Resultados que llevaron a la conclusión

Cadena de hash, veredicto STAT, comparación legacy y criterios congelados del `stage_g0_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('G0', 'stages/stage_g0_qc.json'):
        q = nb.load_qc('stages/stage_g0_qc.json', RUN_ID)
        print('hash_chain_ok:', q['hash_chain_ok'], '| status:', q['hash_chain']['status'])
        sv = q['stat_verdict']
        print(f"stat_verdict: {sv['status']} (usable={sv['usable']}, factor {sv['factor']}) -> ruido empírico")
        lc = q['legacy_comparison']
        print(f"legacy: {lc['n_flagged']}/{lc['n_bands']} bandas flagged vs {lc['legacy_run']} (documentario)")
        print('frozen_criteria_untouched:', q['frozen_criteria_untouched'])
        print(f"cubo: {q['input_cube']['file'].split('/')[-1]} (entry_point={q['input_cube']['entry_point']}, NaN {q['input_cube']['nan_fraction_data']:.3f})")
        print('\nopen_issues:')
        for a in q['open_issues']:
            s = a['issue'] if isinstance(a, dict) else a
            print('  -', s[:100])


## Plot — comparación con el legacy (ADP)

Razón del flujo integrado por banda (realineado / legacy ADP), del `g0_legacy_comparison.csv`. Rojo = flagged. Es **documentario**: donde el continuo es negativo (p.ej. post-Hα) la razón no es fiable — la diferencia principal es la calibración de flujo entre reducciones, esperada.


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    rd = nb.run_dir(RUN_ID)
    lc = nb.load_qc('stages/stage_g0_qc.json', RUN_ID).get('legacy_comparison') or {}
    if not lc.get('table'):
        raise FileNotFoundError('sin comparación legacy para este objeto: '
                                + str(lc.get('reason') or 'no hay tabla declarada en el QC'))
    d = pd.read_csv(rd / 'tables' / 'g0_legacy_comparison.csv')
    x = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(10, 4.3))
    ax.bar(x, d['ratio_new_over_legacy'], 0.55, color=['tab:red' if f else 'tab:green' for f in d['flagged']])
    ax.axhline(1.0, color='k', ls='--', lw=1, label='ratio=1 (idéntico)'); ax.axhline(0, color='0.6', lw=0.6)
    ax.set_xticks(x); ax.set_xticklabels(d['band'], rotation=30, ha='right', fontsize=8)
    for i, r in enumerate(d['ratio_new_over_legacy']):
        ax.text(i, r + 0.05 * np.sign(r), f'{r:.2f}', ha='center', fontsize=7)
    ax.set_ylabel('flujo realineado / legacy (ADP)')
    ax.set_title(f"G0 · comparación con legacy ADP: {int(d['flagged'].sum())}/{len(d)} bandas flagged (documentario)")
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'g0_real_cube'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'legacy_comparison.png', dpi=110); print('figura ->', outdir / 'legacy_comparison.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- G0 cerrado **retroactivamente** (`hash_chain_ok=True`, criterios congelados intactos); ejecución real-cube verificada. · [`docs/g0_execution_log.md`](../docs/g0_execution_log.md)
- Comparación legacy = calibración de flujo distinta entre reducciones (esperado); 3/9 bandas flagged, documentario (continuo negativo). `n/d` = sin ADP de archivo para este objeto.
- M5 STAT red → ruido empírico aguas abajo (consistente con A4). · [`docs/noise_model.md`](../docs/noise_model.md)


## Conclusión (registrada)

**G0: cadena real-cube ejecutada y verificada; `hash_chain_ok = True`, criterios congelados intactos = True.**

- **Proveniencia:** hash_chain sobre el cubo declarado en el QC de este objeto.
- **STAT:** red (M5, factor 4.26) → ruido empírico.
- **Legacy:** 3 de 9 bandas flagged vs `ROXs12b_B_adp`, documentario (calibración de flujo distinta); `n/d` = sin ADP de archivo para este objeto.
- **Deviación honesta:** reproducido retroactivamente vía `run_g0.py`, no en rama fresca — cumplido en sustancia.
- **Downstream:** entrada a G1 (validación de extracción), G2–G5.
